In [ ]:
import glob
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import seaborn as sns
import marsilea as ma
import marsilea.plotter as mp
import scienceplots
# %matplotlib widget
import pickle
plt.style.use(['science', 'nature'])
import numpy as np
from itertools import combinations, product
import pingouin as pg
from matplotlib.patches import Rectangle
from marsilea.plotter import FixedChunk
from matplotlib.colors import LinearSegmentedColormap
plt.rcParams['xtick.labelsize'] = 5
plt.rcParams['ytick.labelsize'] = 5
plt.rcParams['axes.labelsize'] = 6
plt.rcParams["xtick.top"] = False
plt.rcParams["ytick.right"] = False
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["legend.fontsize"] = 6
plt.rcParams['hatch.linewidth'] = 0.5

# %matplotlib widget
tool_map = {
    "scapa": "scAPA",
    "scapatrap": "scAPAtrap",
    "sierra": "Sierra",
    "maaper": "MAAPER",
    "scapture": "SCAPTURE",
    "scape": "SCAPE",
    "infernape": "Infernape",
}

protocol_map = {
    "Visium": "10X Visium",
    "VisiumHD": "10X Visium HD",
    "Chromium": "10X Chromium",
    "Dropseq": "Drop-seq",
    "Stereoseq": "Stereo-seq",
    "Slideseq": "Slide-seq V2",
    "SpatialTranscriptomics": "ST",
    "Microwell": "Microwell-seq",
}

protocol_order = ["10X Chromium", "Drop-seq", "Microwell-seq", "10X Visium","Stereo-seq", "Slide-seq V2", "ST"]
tool_order = ["scAPAtrap","SCAPE", "Infernape", "SCAPTURE", "scAPA",  "Sierra"]

# order = ["10X Chromium", "Drop-seq", "Microwell-seq", "10X Visium", "10X Visium HD","Stereo-seq", "Slide-seq V2", "Spatial Transcriptomics"]

color = [
    "#386b98",
    "#269a51",
    "#edaa4d",
    "#d34123",
    "#7e648a",
    "#454545",
    "#929292",
]
palette=sns.color_palette(color, 7)
mm = 1/25.4

In [ ]:
# pas_quantification_performance_df.to_csv('../../data/performance/pas_quantification_performance.tsv', sep='\t', index=False)
pas_quantification_performance_df = pd.read_csv('../../data/result/merged_tool_performance/pas_quantification_performance.tsv', sep='\t')
pas_quantification_performance_df['tool'] = pas_quantification_performance_df['tool'].map(tool_map)
pas_quantification_performance_df['protocol'] = pas_quantification_performance_df['protocol'].map(protocol_map)

In [ ]:
# 处理离群值，去除分位数0.999以上的值
pas_quantification_performance_df = pas_quantification_performance_df[pas_quantification_performance_df["mape_pas"] <= pas_quantification_performance_df["mape_pas"].quantile(0.999)]
pas_quantification_performance_df = pas_quantification_performance_df[pas_quantification_performance_df["mape_pas_ct"] <= pas_quantification_performance_df["mape_pas_ct"].quantile(0.999)]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 转换工具名称

# 设置绘图样式
# sns.set_theme(style="whitegrid", font_scale=1.2)
# 删除x的minor tick
plt.figure(figsize=(75*mm, 100*mm))
plt.rcParams['xtick.minor.visible'] = False

outlier_params = {
    'flierprops': {
        'marker': 'o',
        'markersize': 0.1,  # 调整点大小
        'markerfacecolor': 'gray',
        'markeredgecolor': 'gray'
    }
}

# 遍历三个评估指标
metrics = ['cor_pas', 'mape_pas', 'mape_pas_ct']
for i, metric in enumerate(metrics, 1):
    plt.subplot(3, 1, i)
    
    # 绘制箱线图
    ax = sns.boxplot(
        data=pas_quantification_performance_df,
        x="tool",
        y=metric,
        hue="match_type",
        order=tool_order,
        palette=palette,
        dodge=True,
        width=0.8,
        **outlier_params
    )
    ax.grid(linestyle='--', alpha=0.6, linewidth=0.5)
    # 设置坐标轴和标签
    # ax.set_title(metric.capitalize(), fontsize=6, weight='bold')
    ax.set_xlabel("", fontsize=6)
    ax.set_ylabel(metric.capitalize(), fontsize=6)
    # ax.tick_params(axis='x', rotation=45)
    
    if i == 1:
        handles, labels = ax.get_legend_handles_labels()
        # 将图例定位在整张图片右侧
        ax.legend(
            handles,
            labels,
            title="Match Type",
            title_fontsize=6,
            fontsize=6,
            frameon=False,
            bbox_to_anchor=(1.02, 0.5),  # 向右移动系数
            loc='center left',
            borderaxespad=0.
        )
    else:
        ax.get_legend().remove()

    if i == 3:
        ax.set_xlabel("Tool", fontsize=6)
    else:
        ax.set_xlabel("")
        ax.set_xticklabels([])

plt.savefig('../../figures/pas_quantification_performance.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 数据预处理
# match_performance_df['tool'] = match_performance_df['tool'].map(tool_map)
# match_performance_df['protocol'] = match_performance_df['protocol'].map(protocol_map)

# 全局样式设定
# plt.rcParams.update({'font.size': 7})
PALETTE = palette
flierprops = dict(marker='o', markersize=0.5, markerfacecolor='gray', markeredgecolor='gray')

metric_map = {
    'cor_pas': 'Correlation',
    'mape_pas': 'MAPE (barcode level)',
    'mape_pas_ct': 'MAPE (group level)'
}

# 遍历每个指标绘图
for metric in ['cor_pas', 'mape_pas', 'mape_pas_ct']:
    # 创建新画布，设定成A4纵向纸张级别的高度
    fig, axes = plt.subplots(
        nrows=len(protocol_order), 
        ncols=1,
        figsize=(60*mm, 170*mm),  # 宽180mm, 高280mm转英寸
        dpi=300
    )
    
    # 主标题（可选）
    # fig.suptitle(f"{metric.capitalize()} Across Protocols", y=0.98, fontsize=9)
    
    # 遍历每个protocol画子图
    for idx, (protocol, ax) in enumerate(zip(protocol_order, axes)):
        # 数据筛选
        plot_data = pas_quantification_performance_df[
            (pas_quantification_performance_df['protocol'] == protocol) &
            (pas_quantification_performance_df[metric].notna())
        ]
        
        # 绘制箱线图核心
        sns.boxplot(
            data=plot_data,
            x="tool",
            y=metric,
            hue="match_type",
            order=tool_order,
            palette=PALETTE,
            ax=ax,
            dodge=True,
            width=0.7,
            flierprops=flierprops,
            linewidth=0.4
        )
        
        # 子图装饰
        ax.set_title(f"{protocol}", loc='left', pad=2, fontsize=6)
        ax.set_xlabel("")  # 隐藏所有x轴标签
        ax.set_ylabel(metric_map[metric] if idx == 0 else "", labelpad=2)  # 仅第一列显示y标签
        ax.tick_params(axis='both', labelsize=6, pad=1)
        ax.grid(True, linestyle=':', linewidth=0.3)
        
        # 隐藏除最后一行外的x轴刻度标签
        if idx != len(protocol_order)-1:
            ax.set_xticklabels([])
        else:
            plt.xticks(fontsize=6, rotation=90)
        
        # 移除单个子图图例
        leg = ax.get_legend()
        if leg:
            leg.remove()

    # 全图统一定制图例
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc='upper center',
        bbox_to_anchor=(0.5, 0.95),  # 定位到画布上方
        ncol=len(labels),            # 关键：水平排列列数等于类别数
        title='Match Type',
        frameon=False,
        title_fontsize=7,
        fontsize=6,
        columnspacing=0.8,          # 控制图例项间距
        handletextpad=0.3           # 控制符号与文本间距
    )

    # # 调整整体布局
    # plt.subplots_adjust(
    #     hspace=0.25,  # 垂直间距控制
    #     left=0.12,    # 左边界
    #     right=0.88,   # 右边界（需根据图例调整）
    #     bottom=0.08   # 底边界
    # )
    
    
    plt.savefig(f'../../figures/suppfig/pas_quantification_{metric}_by_protocol.pdf', dpi=300, bbox_inches='tight')
    plt.show()  # 立即显示当前figure
    plt.close()  # 关闭当前figure释放内存
